In [1]:
"""
Inférence libre des modèles OpenRouter sur la feuille OPEN QUESTION.
-------------------------------------------------------------------
- Aucune contrainte de format, aucun parsing : on récupère la réponse
  brute du modèle telle quelle.
- Aucune limite de tokens (max_tokens non envoyé) : les modèles répondent
  librement.
- Une seule colonne ajoutée par modèle : response_<nom_court>.
- Logs dynamiques (en continu) du nombre de tokens dépensé par modèle,
  cumulé, pour ne pas avoir à faire la somme soi-même.
- Reprise automatique : ré-exécuter la cellule reprend là où ça s'est
  arrêté et re-remplit les cellules vides OU en erreur (jamais celles
  déjà répondues correctement).

À vérifier avant de lancer : EXCEL_PATH, SHEET_NAME, OPENROUTER_API_KEY, MODEL_MAP.
"""

import os
import time
import random
import threading
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd
import requests

# ============================== CONFIG ==============================

EXCEL_PATH = "open_question_final_no_responses.xlsx"
SHEET_NAME = "Feuil1"   # <-- vérifie le nom exact de ta feuille

OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY", "sk-or-v1-cf1b6e73b11ed")
API_URL = "https://openrouter.ai/api/v1/chat/completions"

# nom complet OpenRouter -> nom court utilisé dans la colonne de réponse
MODEL_MAP = {
    "google/gemma-4-26b-a4b-it": "gemma4_26b",
    "qwen/qwen3.6-27b": "qwen3_6_27b",
    "qwen/qwen3.5-9b": "qwen3_5_9b",
    "cohere/command-r-08-2024": "commandr",
    "google/gemma-4-31b-it": "gemma4_31b",
    "openai/gpt-oss-20b": "gptoss_20b",
    "openai/gpt-5": "gpt5",
    "anthropic/claude-sonnet-5": "claude_sonnet5",
    "google/gemini-3.6-flash": "gemini3_6_flash",
    "x-ai/grok-4.20": "grok4_20",
    "mistralai/mistral-small-3.2-24b-instruct": "mistral_24b",
    "mistralai/mistral-saba": "mistral_saba",
}

# --- concurrence / robustesse ---
MAX_WORKERS = 10
MAX_RETRIES = 4
RETRY_BACKOFF_SECONDS = 5
REQUEST_TIMEOUT = 300      # génération libre = réponses longues, timeout large
SAVE_EVERY = 25           # sauvegarde toutes les N lignes terminées
JITTER_BEFORE_CALL = 0.15

# Valeurs considérées "à (re)faire" au moment de la reprise
ERROR_PREFIXES = ("[ERROR]", "[UNEXPECTED_ERROR]", "[PROMPT_ERROR]")

# ============================== PROMPT ==============================

def build_prompt(row) -> str:
    """Inférence libre : on passe juste la question, avec une consigne
    minimale de langue (les questions sont en darija) + une consigne
    de concision."""
    return (
        f"{row['input']}\n\n"
        "جاوب بالدارجة المغربية ولا بالعربية الفصحى. "
        "كون مختصر ومباشر فجوابك، بلا حشو."
    )

# ============================== SUIVI TOKENS ==============================

# Renseigné et lu uniquement dans le thread principal (boucle as_completed),
# donc pas besoin de lock.
TOKEN_USAGE = {}  # model_short -> {"in":.., "out":.., "total":.., "calls":..}

def reset_usage(model_short):
    TOKEN_USAGE[model_short] = {"in": 0, "out": 0, "total": 0, "calls": 0}

def add_usage(model_short, usage):
    p = int(usage.get("prompt_tokens", 0) or 0)
    c = int(usage.get("completion_tokens", 0) or 0)
    t = int(usage.get("total_tokens", 0) or (p + c))
    u = TOKEN_USAGE[model_short]
    u["in"] += p
    u["out"] += c
    u["total"] += t
    u["calls"] += 1
    return p, c, t

# ============================== APPEL API ==============================

REASONING_MODE = {}
_reasoning_lock = threading.Lock()
_thread_local = threading.local()

import re
REASONING_MANDATORY_PAT = re.compile(r"reasoning is mandatory", re.IGNORECASE)

def get_session() -> requests.Session:
    s = getattr(_thread_local, "session", None)
    if s is None:
        s = requests.Session()
        adapter = requests.adapters.HTTPAdapter(pool_connections=MAX_WORKERS,
                                                pool_maxsize=MAX_WORKERS)
        s.mount("https://", adapter)
        _thread_local.session = s
    return s

def get_mode(model: str) -> str:
    with _reasoning_lock:
        return REASONING_MODE.setdefault(model, "off")

def set_mode(model: str, mode: str):
    with _reasoning_lock:
        if REASONING_MODE.get(model) != mode:
            REASONING_MODE[model] = mode
            print(f"  >>> [{model}] bascule reasoning en '{mode}'")

def build_payload(model: str, prompt: str, mode: str) -> dict:
    # PAS de max_tokens : le modèle répond librement.
    payload = {
        "model": model,
        "messages": [{"role": "user", "content": prompt}],
        "usage": {"include": True},  # demande explicitement le décompte de tokens
    }
    if mode == "off":
        payload["reasoning"] = {"enabled": False}
    elif mode == "on":
        payload["reasoning"] = {"enabled": True}
    # mode == "omit" -> champ reasoning absent
    return payload

def call_openrouter(model: str, prompt: str):
    """Retourne (contenu, usage_dict, erreur). Un seul de contenu/erreur est non-None."""
    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json",
    }
    session = get_session()
    last_error = None
    tried_modes = set()

    for attempt in range(1, MAX_RETRIES + 1):
        mode = get_mode(model)
        tried_modes.add(mode)
        try:
            resp = session.post(API_URL, headers=headers,
                                json=build_payload(model, prompt, mode),
                                timeout=REQUEST_TIMEOUT)
        except requests.exceptions.RequestException as e:
            last_error = f"request_exception: {e}"
            time.sleep(RETRY_BACKOFF_SECONDS * attempt + random.uniform(0, 1))
            continue

        if resp.status_code == 200:
            try:
                data = resp.json()
                msg = data["choices"][0]["message"]
                content = msg.get("content")
                if not content:  # certains modèles mettent tout dans "reasoning"
                    content = msg.get("reasoning") or ""
                usage = data.get("usage", {}) or {}
                return content, usage, None
            except Exception as e:
                return None, {}, f"unexpected_response_format: {e} | raw={resp.text[:300]}"

        body = resp.text[:500]

        if resp.status_code in (400, 404, 422) and REASONING_MANDATORY_PAT.search(body):
            set_mode(model, "on")
            continue
        if resp.status_code in (400, 404, 422) and "reasoning" in body.lower():
            if "omit" not in tried_modes:
                set_mode(model, "omit")
                continue

        if resp.status_code == 429:
            last_error = f"rate_limited (429): {body[:200]}"
            time.sleep(RETRY_BACKOFF_SECONDS * attempt * 2 + random.uniform(0, 2))
        elif 500 <= resp.status_code < 600:
            last_error = f"server_error ({resp.status_code}): {body[:200]}"
            time.sleep(RETRY_BACKOFF_SECONDS * attempt + random.uniform(0, 1))
        else:
            return None, {}, f"http_{resp.status_code}: {body[:300]}"

    return None, {}, last_error

# ============================== I/O EXCEL ==============================

def load_all_sheets(path: str) -> dict:
    if not Path(path).exists():
        raise FileNotFoundError(f"Fichier introuvable : {path}")
    return pd.read_excel(path, sheet_name=None)

def save_all_sheets(sheets: dict, path: str):
    p = Path(path)
    tmp_path = p.with_name(p.stem + ".tmp" + p.suffix)
    with pd.ExcelWriter(tmp_path, engine="openpyxl") as writer:
        for name, sheet_df in sheets.items():
            sheet_df.to_excel(writer, sheet_name=name, index=False)
    tmp_path.replace(path)

# ============================== REPRISE ==============================

def needs_processing(val) -> bool:
    """True si la cellule est vide, 'EMPTY', ou en erreur -> à (re)faire."""
    if pd.isna(val):
        return True
    s = str(val).strip()
    if s == "" or s == "EMPTY":
        return True
    if s.startswith(ERROR_PREFIXES):
        return True
    return False

# ============================== BOUCLE PRINCIPALE ==============================

sheets = None
df = None

def _work(model_full, idx, prompt):
    """Thread : ne touche jamais au dataframe."""
    if JITTER_BEFORE_CALL:
        time.sleep(random.uniform(0, JITTER_BEFORE_CALL))
    content, usage, error = call_openrouter(model_full, prompt)
    return idx, content, usage, error

def run_inference():
    global sheets, df
    sheets = load_all_sheets(EXCEL_PATH)
    df = sheets[SHEET_NAME]
    if "input" not in df.columns:
        raise ValueError("Colonne 'input' introuvable dans la feuille.")

    for model_full, model_short in MODEL_MAP.items():
        col = f"response_{model_short}"
        if col not in df.columns:
            df[col] = pd.NA

        reset_usage(model_short)

        todo = [i for i in df.index if needs_processing(df.at[i, col])]
        total_todo = len(todo)
        print(f"\n=== {model_full} ({model_short}) — {total_todo} ligne(s) à faire ===")
        if not todo:
            continue

        done = 0
        done_since_save = 0
        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
            futures = {}
            for idx in todo:
                try:
                    prompt = build_prompt(df.loc[idx])
                except Exception as e:
                    df.at[idx, col] = f"[PROMPT_ERROR] {e}"
                    print(f"  [{idx}] prompt impossible : {e}")
                    continue
                futures[pool.submit(_work, model_full, idx, prompt)] = idx

            for fut in as_completed(futures):
                idx = futures[fut]
                item_id = df.at[idx, "item_id"] if "item_id" in df.columns else idx
                try:
                    idx, content, usage, error = fut.result()
                    if error is not None:
                        df.at[idx, col] = f"[ERROR] {error}"
                        print(f"  [{item_id}] ERREUR API: {error}")
                    else:
                        df.at[idx, col] = content if content else "EMPTY"
                        p, c, t = add_usage(model_short, usage)
                        u = TOKEN_USAGE[model_short]
                        done_disp = done + 1
                        print(
                            f"  [{item_id}] +{t:,} tok "
                            f"| cumul {model_short}: {u['total']:,} tok "
                            f"(in {u['in']:,} / out {u['out']:,}) "
                            f"· {u['calls']} appels · {done_disp}/{total_todo}"
                        )
                except Exception as e:
                    df.at[idx, col] = f"[UNEXPECTED_ERROR] {e}"
                    print(f"  [{item_id}] ERREUR INATTENDUE: {e}")

                done += 1
                done_since_save += 1
                if done_since_save >= SAVE_EVERY:
                    try:
                        sheets[SHEET_NAME] = df
                        save_all_sheets(sheets, EXCEL_PATH)
                    except Exception as e:
                        print(f"  (sauvegarde échouée, on continue: {e})")
                    done_since_save = 0

        u = TOKEN_USAGE[model_short]
        print(f"  --- {model_short} terminé : {u['total']:,} tokens "
              f"(in {u['in']:,} / out {u['out']:,}) sur {u['calls']} appels ---")
        try:
            sheets[SHEET_NAME] = df
            save_all_sheets(sheets, EXCEL_PATH)
        except Exception as e:
            print(f"Sauvegarde finale échouée pour {model_short}: {e}")

    print("\n=== Récap tokens (cette exécution) ===")
    for model_short, u in TOKEN_USAGE.items():
        print(f"  {model_short:>18} : {u['total']:,} tok "
              f"(in {u['in']:,} / out {u['out']:,}) · {u['calls']} appels")
    print("\nTerminé.")


try:
    run_inference()
except KeyboardInterrupt:
    print("\nInterrompu — résultats déjà obtenus sauvegardés. "
          "Ré-exécute la cellule pour reprendre.")


=== google/gemma-4-26b-a4b-it (gemma4_26b) — 0 ligne(s) à faire ===

=== qwen/qwen3.6-27b (qwen3_6_27b) — 0 ligne(s) à faire ===

=== qwen/qwen3.5-9b (qwen3_5_9b) — 0 ligne(s) à faire ===

=== cohere/command-r-08-2024 (commandr) — 0 ligne(s) à faire ===

=== google/gemma-4-31b-it (gemma4_31b) — 0 ligne(s) à faire ===

=== openai/gpt-oss-20b (gptoss_20b) — 0 ligne(s) à faire ===

=== openai/gpt-5 (gpt5) — 0 ligne(s) à faire ===

=== anthropic/claude-sonnet-5 (claude_sonnet5) — 0 ligne(s) à faire ===

=== google/gemini-3.6-flash (gemini3_6_flash) — 401 ligne(s) à faire ===
  >>> [google/gemini-3.6-flash] bascule reasoning en 'on'
  [OQ_577] +831 tok | cumul gemini3_6_flash: 831 tok (in 73 / out 758) · 1 appels · 1/401
  [OQ_570] +960 tok | cumul gemini3_6_flash: 1,791 tok (in 157 / out 1,634) · 2 appels · 2/401
  [OQ_569] +968 tok | cumul gemini3_6_flash: 2,759 tok (in 231 / out 2,528) · 3 appels · 3/401
  [OQ_584] +1,074 tok | cumul gemini3_6_flash: 3,833 tok (in 321 / out 3,512) · 4 a

In [2]:
1

1

5

In [6]:
5

5

In [ ]:
hh

In [ ]:
5

In [1]:
5

5